# Modèle de production — GBS v7 + θ figé

Ce notebook charge le modèle v7 et **intègre la règle de décision** :

```
Pour chaque éclair :
   conf(xi) = S(30|xi)

   Si conf(xi) > θ = 0.30  ─►  PRÉDICTION ACTIVE
                                └─ propose de lever à t + T*_i

   Si conf(xi) ≤ θ = 0.30  ─►  PRÉDICTION INACTIVE
                                └─ rien à faire, règle 30 min reste en vigueur
```

**θ = 0,30** a été calibré dans `calibration_theta_test.ipynb` sur les données TEST 2021-2022, **figé** ici.

Le notebook expose une **fonction unique** `predict_alerte(eclairs)` qu'on peut appeler sur n'importe quelle alerte (un ou plusieurs éclairs) et qui rend une décision opérationnelle.

## 1. Setup et chargement du modèle

In [ ]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=FutureWarning)

ROOT       = Path('..').resolve()
MODELS_DIR = ROOT / 'models'

# ╔════════════════════════════════════════════════════════════╗
# ║  CONFIGURATION FIGÉE — résultat de la calibration sur TEST ║
# ╠════════════════════════════════════════════════════════════╣
MODEL_VERSION = 'v7'
THETA         = 0.30   # ← seuil figé après calibration test 2021-2022
RATIO         = 0.98   # T*_i = min{t : S(t) ≤ S(30)/0.98}
MAX_GAP_MIN   = 30     # règle baseline
# ╚════════════════════════════════════════════════════════════╝

FEATURES = [
    'h_cos','h_sin','doy_cos','doy_sin','saison',
    'dist_centre','dist_avg_5','dist_min_so_far',
    'silence_min','freq_5min','rang','rang_norm',
    'airport_enc',
]

gbs    = joblib.load(MODELS_DIR / f'gbs_{MODEL_VERSION}_model.pkl')
scaler = joblib.load(MODELS_DIR / f'gbs_{MODEL_VERSION}_scaler.pkl')
le     = joblib.load(MODELS_DIR / f'gbs_{MODEL_VERSION}_label_encoder.pkl')

print(f'✓ Modèle GBS {MODEL_VERSION} chargé')
print(f'✓ θ figé = {THETA}')
print(f'✓ Aéroports : {list(le.classes_)}')
print(f'✓ Features  : {len(FEATURES)}')

## 2. Feature engineering

In [ ]:
def build_features(df):
    """Construit les 13 features sur un DataFrame d'éclairs.

    Colonnes minimales requises : date, airport, airport_alert_id, dist
    """
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'], utc=True)
    df = df.sort_values(['airport','airport_alert_id','date']).reset_index(drop=True)
    g   = df.groupby(['airport','airport_alert_id'])
    h   = df['date'].dt.hour + df['date'].dt.minute / 60
    doy = df['date'].dt.dayofyear
    df['h_cos']   = np.cos(2*np.pi*h/24)
    df['h_sin']   = np.sin(2*np.pi*h/24)
    df['doy_cos'] = np.cos(2*np.pi*doy/365)
    df['doy_sin'] = np.sin(2*np.pi*doy/365)
    df['saison']  = ((df['date'].dt.month % 12)//3)+1
    prev = g['date'].shift(1)
    df['silence_min'] = ((df['date']-prev).dt.total_seconds()/60).fillna(30).clip(0,60)
    df['freq_5min']   = (1/df['silence_min'].clip(lower=0.5)).clip(upper=10)
    df['dist_centre']     = df['dist']
    df['dist_avg_5']      = g['dist'].transform(lambda x: x.rolling(5,min_periods=1).mean())
    df['dist_min_so_far'] = g['dist'].cummin()
    df['rang']            = g.cumcount()
    df['rang_norm']       = df['rang']/g['rang'].transform('max').clip(lower=1)
    df['airport_enc']     = le.transform(df['airport'])
    for col in FEATURES:
        if df[col].isna().any():
            df[col] = df[col].fillna(df[col].median())
    return df

## 3. ⭐ Cœur de la décision — `predict_eclair()`

Pour **chaque éclair** indépendamment :
1. Calcule `conf` et `T*_i`
2. Si `conf > θ` → renvoie `(actif=True, t_recommande)`
3. Si `conf ≤ θ` → renvoie `(actif=False, None)` — règle 30 min appliquée par défaut

In [ ]:
def predict_eclairs(df_eclairs):
    """Calcule conf et T*_i pour chaque éclair, applique θ figé.

    Renvoie le DataFrame enrichi avec :
      - confiance        : S(30|xi)
      - horizon_min      : T*_i (minutes)
      - modele_actif     : bool (conf > θ)
      - decision_eclair  : str (texte humain)
      - fin_predite      : timestamp si modele_actif, sinon NaT
    """
    df = build_features(df_eclairs)
    X  = scaler.transform(df[FEATURES].values.astype(float))
    surv_fns = gbs.predict_survival_function(X)

    s30     = np.array([float(fn(MAX_GAP_MIN)) for fn in surv_fns])
    t_stars = np.full(len(surv_fns), float(MAX_GAP_MIN))
    for i, (fn, s) in enumerate(zip(surv_fns, s30)):
        if s <= 0:
            continue
        idx = np.searchsorted(-fn.y, -(s/RATIO), side='left')
        if idx < len(fn.x):
            t_stars[i] = min(float(fn.x[idx]), float(MAX_GAP_MIN))

    df['confiance']    = s30
    df['horizon_min']  = t_stars
    df['modele_actif'] = df['confiance'] > THETA
    df['fin_predite']  = pd.NaT
    df.loc[df['modele_actif'], 'fin_predite'] = (
        df.loc[df['modele_actif'], 'date']
        + pd.to_timedelta(df.loc[df['modele_actif'], 'horizon_min'], unit='m')
    )
    df['decision_eclair'] = df['modele_actif'].map({
        True:  '✓ ACTIF — propose levée',
        False: '⊘ INACTIF — règle 30 min',
    })
    return df

## 4. ⭐ Décision globale pour une alerte — `predict_alerte()`

Combine les décisions éclair-par-éclair pour décider quand lever l'alerte :

- **Si au moins un éclair confiant** → `t_alerte = min(fin_predite des confiants)`  
- **Sinon** → `t_alerte = dernier_eclair + 30 min` (règle baseline)

In [ ]:
def predict_alerte(df_eclairs):
    """Décision globale pour une (ou plusieurs) alerte(s).

    Renvoie un dict par alerte avec :
      - t_regle           : dernier éclair + 30 min  (baseline Météorage)
      - t_modele          : décision modèle si au moins 1 confiant, sinon = t_regle
      - gain_min          : minutes économisées vs règle
      - n_eclairs_actifs  : nombre d'éclairs qui ont voté
      - decision          : 'MODELE' ou 'REGLE_30_MIN'
    """
    scored = predict_eclairs(df_eclairs)
    out    = []
    for (airport, alert_id), grp in scored.groupby(['airport','airport_alert_id']):
        t_regle = grp['date'].max() + pd.Timedelta(minutes=MAX_GAP_MIN)
        actifs  = grp[grp['modele_actif']]
        if len(actifs):
            t_modele = actifs['fin_predite'].min()
            decision = 'MODELE'
        else:
            t_modele = t_regle
            decision = 'REGLE_30_MIN'
        gain_min = max((t_regle - t_modele).total_seconds()/60, 0)
        out.append({
            'airport':          airport,
            'airport_alert_id': int(alert_id),
            'n_eclairs':        len(grp),
            'n_actifs':         int(grp['modele_actif'].sum()),
            't_regle':          t_regle,
            't_modele':         t_modele,
            'gain_min':         round(gain_min, 1),
            'decision':         decision,
        })
    return pd.DataFrame(out), scored

## 5. Exemple — alerte synthétique

On simule une alerte sur Bastia avec 5 éclairs, dont les derniers s'éloignent.

In [ ]:
exemple = pd.DataFrame({
    'airport':          ['Bastia']*5,
    'airport_alert_id': [999]*5,
    'date': pd.to_datetime([
        '2026-07-15 12:00:00',
        '2026-07-15 12:03:00',
        '2026-07-15 12:08:00',
        '2026-07-15 12:14:00',
        '2026-07-15 12:21:00',
    ], utc=True),
    'dist': [18.0, 12.0, 9.0, 14.0, 22.0],
})

decisions, scored = predict_alerte(exemple)

print('── DÉTAIL PAR ÉCLAIR ──')
cols = ['date','airport','dist','confiance','horizon_min','modele_actif','decision_eclair']
print(scored[cols].to_string(index=False))

print('\n── DÉCISION GLOBALE ──')
print(decisions.to_string(index=False))

## 6. Application sur un jeu réel

Exemple : toutes les alertes du jury 2023-2025.

In [ ]:
jury_path = ROOT / 'segment_alerts_all_airports_eval.csv'
if jury_path.exists():
    df_jury = pd.read_csv(jury_path)
    df_jury = df_jury[df_jury['alert_id'].notna()].copy()
    df_jury = df_jury.rename(columns={'alert_id':'airport_alert_id'})
    print(f'Jury chargé : {len(df_jury):,} éclairs')

    decisions_jury, _ = predict_alerte(df_jury)

    n_modele = (decisions_jury['decision'] == 'MODELE').sum()
    n_regle  = (decisions_jury['decision'] == 'REGLE_30_MIN').sum()
    gain_h   = decisions_jury['gain_min'].sum() / 60

    print(f'\nDécisions sur {len(decisions_jury)} alertes :')
    print(f'  ✓ MODELE       : {n_modele}  ({100*n_modele/len(decisions_jury):.0f} %)')
    print(f'  ⊘ REGLE_30_MIN : {n_regle}  ({100*n_regle/len(decisions_jury):.0f} %)')
    print(f'  Gain total     : {gain_h:.2f} h')
    print(f'  Gain moyen     : {decisions_jury["gain_min"].mean():.2f} min/alerte')

    decisions_jury.head(20)
else:
    print('Jury non trouvé localement.')

## 7. Sauvegarde du modèle de production (artefact unique)

On regroupe modèle + scaler + label encoder + θ dans **un seul fichier** déployable.

In [ ]:
artifact = {
    'model':          gbs,
    'scaler':         scaler,
    'label_encoder':  le,
    'theta':          THETA,
    'ratio':          RATIO,
    'max_gap_min':    MAX_GAP_MIN,
    'features':       FEATURES,
    'version':        f'gbs_{MODEL_VERSION}_production',
    'calibration':    'theta=0.30 calibré sur TEST 2021-2022, sous contrainte Risk<2%',
    'evaluation':     'Sur jury 2023-2025 : Gain=103.5h, Risk=0.10%, 2/1995 manqués <3km',
}
out_path = MODELS_DIR / f'gbs_{MODEL_VERSION}_production.pkl'
joblib.dump(artifact, out_path)
print(f'✓ Modèle de production sauvé : {out_path.name}')
print(f'  ({out_path.stat().st_size/1024:.1f} Ko)')

## 8. Comment utiliser en production

Depuis n'importe quel script ou notebook :

```python
import joblib
art = joblib.load('models/gbs_v7_production.pkl')

# … construire df_eclairs (date, airport, airport_alert_id, dist)
# … puis appliquer le même feature engineering et la décision intégrée :

# 1) features
X = scaler.transform(df_features[art['features']].values)
# 2) survies
fns = art['model'].predict_survival_function(X)
# 3) confiance
conf = np.array([fn(art['max_gap_min']) for fn in fns])
# 4) décision binaire
actif = conf > art['theta']        # ← LE SEUIL EST DANS L'ARTIFACT
# 5) si pas actif → règle 30 min ; sinon T*_i et min sur l'alerte
```

Le seuil est désormais **figé dans l'artefact**, plus de risque qu'il change accidentellement. Toute modification de θ nécessite une nouvelle calibration explicite.